# Phase 3 Legal Rhetorical Role Prediction with the Phase 1 V2 checkpoint

This notebook adapts the **already trained V2 encoder–decoder model** to the 13 rhetorical roles in the supplied Corporate Law (CL) and Information Technology (IT) data. It does **not** rerun Phase 1 pretraining and does not download another backbone. It strictly loads `v2_model.pt`, keeps the original V2 tokenizer and model definitions, then trains Phase 3 task components.

The V2 source notebook, checkpoint, tokenizer, label vocabulary, and six CL/IT JSON files are discovered under `/kaggle/input`. Attach a Kaggle dataset containing those files before running. No Hugging Face login or external data download is needed.

## What this run compares

The notebook prepares four submission candidates: **LoRA**, **LoRMA+**, **BitFit**, and **LoRMA+ with task-logit knowledge distillation**. The distillation teacher is a fully fine-tuned V2 encoder trained only on the supplied Phase 3 training split. Each candidate adds the same document-level BiLSTM so it can use neighboring sentence context. The candidate selected by mean CL and IT development Macro-F1 is copied to `/kaggle/working/phase3_rr_v2/submission.zip`. Each candidate ZIP contains one root-level `prediction.json`.

This is the V2 backbone track. It does not complete the separate four-backbone comparison by itself; V1, V3, and V4 need their own Phase 3 runs.


## Evidence from the supplied V2 files

I inspected the original V2 notebook, training history, checkpoint, and Phase 1 `prediction.json` before building this workflow.

| Evidence | Observed result | What it tells us |
|---|---:|---|
| V2 architecture in `v2.ipynb` (configuration cell and model-definition cell) | Encoder–decoder Transformer; 4 encoder layers, 5 decoder layers, width 768, 12 attention heads, learned positions, LayerNorm, tied input/output embeddings | Use the pretrained **encoder** to make sentence representations. The decoder is not needed for classification. |
| Parameter count from the exact V2 source | 95,556,864 | This is a substantial Phase 1 checkpoint, so retraining from scratch would discard the work already done. |
| `training_history.json` | 3,000 optimizer steps; 98,304,000 tokens; train loss 10.2081 → 3.6138 | The language-model objective learned from the Phase 1 corpus. This is not an RR score. |
| Recorded validation loss / perplexity | Validation loss 4.5589 → 3.7451; perplexity 95.48 → 42.32. The last two losses differ by only 0.00009. | Pretraining validation improved and was nearly flat at the end. It does not establish how well V2 labels legal rhetorical roles. |
| Phase 1 `prediction.json` | 1,000 numbered records, each containing generated `text` | These are Phase 1 text continuations, not sentence-role labels, and cannot be submitted for Phase 3. |
| Supplied labeled Phase 3 training data | 80 documents, 16,845 sentences across CL and IT | Enough to train the task head and PEFT adapters without fetching the original Phase 1 corpus. |
| Label counts in combined Phase 3 training data | `RatioOfTheDecision`: 4,301; `PrecedentOverruled`: 3 | The classes are highly imbalanced. The notebook uses capped inverse-square-root weights computed from training labels only and reports every class's development F1. |
| Development split | 5 documents per domain; 2,142 sentences total; several labels absent in CL development | A single small holdout is noisy. Model selection uses a fixed 13-label Macro-F1 for both domains and reports the small support. |
| Token length audit using the original V2 BPE merges | Median is about 33 tokens; 99th percentile is below 200 in all six splits | A 256-token cap retains nearly all sentences while reducing training memory. The notebook prints exact truncation counts for the attached copy. |

The training loop in the V2 notebook evaluated the Phase 1 language-model loss periodically; it did not calculate Phase 3 label metrics. The only Phase 3 quality evidence will come from this notebook's CL/IT development results and, later, the Codabench score. No result score is claimed before those runs happen.


## Design choices and research basis

1. **Reuse V2 instead of starting over.** The Phase 1 checkpoint has a compatible V2 architecture and is strictly loaded before any Phase 3 training. The decoder is frozen and unused because the task is sentence classification.
2. **Pool token states into sentence vectors, then model document order.** The encoder produces one contextual vector per token; masked mean pooling gives one vector per sentence. A bidirectional LSTM then reads the ordered sentence vectors in each judgment. Legal rhetorical-role work has found neighboring-sentence and sequence context useful, which motivates this controlled context layer, but does not guarantee a score increase here [5, 6].
   The workflow is deliberately staged: first adapt the V2 encoder on sentence labels, then cache those representations and train the document BiLSTM with the encoder frozen. This avoids retaining transformer activations for every sentence of a long judgment. It is a memory-conscious Phase 3 experiment, not end-to-end sequence fine-tuning, so a later controlled end-to-end run could test whether joint training helps.
3. **Mask padding explicitly.** The original V2 encoder's `encode` method does not accept a padding mask. This notebook reuses its exact weights and submodules but uses a Phase 3 encoder path that passes a key-padding mask into attention. Otherwise, padded positions in a batch could affect real tokens.
4. **Compare PEFT methods under the same data and selection rule.** LoRA adds a low-rank update to a frozen matrix [1]. LoRMA+ uses the multiplicative update with additive identity/rank inflation described in the ACL Findings paper [2]. BitFit updates encoder bias terms only [3]. The new sentence head and contextual LSTM are trained for all candidates.
5. **Use a task teacher for distillation.** A full encoder-fine-tuned teacher is trained on the Phase 3 training split. The LoRMA+KD student combines gold-label cross-entropy with softened teacher-logit KL loss, following the teacher–student principle [4]. Development labels are only used for selection; test labels are never read.
6. **Control imbalance without letting three examples dominate.** Class weights use inverse square-root frequency, are capped, and are computed from the combined training split only. All candidates use the same weights.
7. **Select fairly across domains.** The selected score is the unweighted mean of CL and IT development Macro-F1, with all 13 label IDs fixed in every calculation. Macro-F1 averages class F1 equally rather than weighting common roles more heavily [7].
8. **Keep the output contract simple.** Predictions are one label string per original test sentence, keyed by document ID and kept in sentence order. The validator checks exact test-document coverage, sentence counts, allowed labels, ZIP CRC, and a single root-level `prediction.json`.

The LoRMA+ update is implemented as `y + (alpha / rank) * B(A(W0 x))`, where `y = W0 x + bias`, `A` maps the output dimension to rank, and `B` maps rank back to the output dimension. `B` starts at zero, so the adapter initially preserves the checkpoint function. This is the paper's additive-inflation form, not ordinary additive LoRA.


## Kaggle setup

1. Enable a GPU accelerator in Kaggle Notebook settings.
2. Attach inputs containing: `v2_model.pt`, `bpe_en_25k.json`, `label_vocab.json`, `CL_train.json`, `CL_dev.json`, `CL_test.json`, `IT_train.json`, `IT_dev.json`, and `IT_test.json`.
3. Run the notebook from top to bottom. It uses `/kaggle/working/phase3_rr_v2/` for outputs.
4. For a GPU out-of-memory error, reduce `BATCH_SIZE` and `ENCODE_BATCH_SIZE` in the configuration cell; retain `MAX_LENGTH = 256` unless you deliberately want to rerun the token-length audit and compare truncation.

The local V2 checkpoint was strictly loaded into this exact source architecture and an encoder forward smoke check passed before this notebook was assembled. The notebook repeats strict loading, verifies parity with V2's original unpadded encoder, and checks padding invariance inside Kaggle.


In [ ]:
import os
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Resolve exact input filenames. If Kaggle has duplicate copies, they must be byte-identical.
ASSET_NAMES = [
    "v2_model.pt", "bpe_en_25k.json", "label_vocab.json",
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
]

def index_assets(root=Path("/kaggle/input")):
    matches = defaultdict(list)
    if root.exists():
        for folder, _, files in os.walk(root):
            for filename in files:
                if filename in ASSET_NAMES:
                    matches[filename].append(Path(folder) / filename)
    return matches

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ASSET_MATCHES = index_assets()

def resolve_asset(filename):
    paths = sorted(ASSET_MATCHES.get(filename, []), key=lambda p: str(p).lower())
    if not paths:
        raise FileNotFoundError(
            f"Missing {filename}. Attach a Kaggle input dataset containing this exact file. "
            f"Searched /kaggle/input."
        )
    if len(paths) > 1:
        hashes = {file_sha256(p) for p in paths}
        if len(hashes) != 1:
            listing = "\n".join(str(p) for p in paths)
            raise RuntimeError(f"Conflicting copies of {filename}:\n{listing}")
    return paths[0]

ASSETS = {name: resolve_asset(name) for name in ASSET_NAMES}
for name, path in ASSETS.items():
    print(f"{name}: {path}")

def read_json(filename):
    with open(ASSETS[filename], "r", encoding="utf-8") as f:
        return json.load(f)

label_to_id = {str(name): int(index) for name, index in read_json("label_vocab.json").items()}
assert len(label_to_id) == 13, f"Expected 13 RR labels, found {len(label_to_id)}"
assert sorted(label_to_id.values()) == list(range(13)), "Label IDs must be contiguous from 0 to 12"
id_to_label = {index: name for name, index in label_to_id.items()}
NUM_LABELS = len(label_to_id)

def normalize_labeled(filename):
    raw = read_json(filename)
    if not isinstance(raw, list):
        raise TypeError(f"{filename} must contain a list of labeled document objects")
    docs = []
    for row in raw:
        if not isinstance(row, dict) or not {"id", "text", "labels"}.issubset(row):
            raise ValueError(f"Bad labeled record in {filename}: expected id, text, labels")
        texts = [str(x) for x in row["text"]]
        labels = [str(x) for x in row["labels"]]
        if len(texts) != len(labels):
            raise ValueError(f"Sentence/label length mismatch in {filename}, id={row['id']}")
        unknown = sorted(set(labels) - set(label_to_id))
        if unknown:
            raise ValueError(f"Unknown labels in {filename}: {unknown}")
        docs.append({"id": str(row["id"]), "text": texts, "labels": labels})
    return docs

def normalize_test(filename):
    raw = read_json(filename)
    docs = []
    if isinstance(raw, dict):
        rows = raw.items()
    elif isinstance(raw, list):
        rows = ((row["id"], row) for row in raw)
    else:
        raise TypeError(f"{filename} must be a document mapping or list")
    for doc_id, value in rows:
        if isinstance(value, dict):
            texts = value.get("text")
        else:
            texts = value
        if not isinstance(texts, list):
            raise ValueError(f"Test document {doc_id} in {filename} has no sentence list")
        docs.append({"id": str(doc_id), "text": [str(x) for x in texts], "labels": None})
    return docs

DOCS = {
    "train": {"CL": normalize_labeled("CL_train.json"), "IT": normalize_labeled("IT_train.json")},
    "dev": {"CL": normalize_labeled("CL_dev.json"), "IT": normalize_labeled("IT_dev.json")},
    "test": {"CL": normalize_test("CL_test.json"), "IT": normalize_test("IT_test.json")},
}

for domain in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        split_ids = [d["id"] for d in DOCS[split][domain]]
        assert len(split_ids) == len(set(split_ids)), f"Duplicate document ID in {domain} {split}"
    train_ids = {d["id"] for d in DOCS["train"][domain]}
    dev_ids = {d["id"] for d in DOCS["dev"][domain]}
    test_ids = {d["id"] for d in DOCS["test"][domain]}
    assert not (train_ids & dev_ids), f"Train/dev document overlap in {domain}"
    assert not (train_ids & test_ids), f"Train/test document overlap in {domain}"
    assert not (dev_ids & test_ids), f"Dev/test document overlap in {domain}"
    print(domain, {split: (len(DOCS[split][domain]), sum(len(d["text"]) for d in DOCS[split][domain])) for split in DOCS})

all_test_ids = [doc["id"] for domain in ("CL", "IT") for doc in DOCS["test"][domain]]
assert len(all_test_ids) == len(set(all_test_ids)), "CL and IT test document IDs must be globally unique"

TRAIN_LABEL_IDS = [label_to_id[y] for domain in ("CL", "IT") for doc in DOCS["train"][domain] for y in doc["labels"]]
TRAIN_LABEL_COUNTS = torch.bincount(torch.tensor(TRAIN_LABEL_IDS), minlength=NUM_LABELS)
print("Training label counts:", {id_to_label[i]: int(TRAIN_LABEL_COUNTS[i]) for i in range(NUM_LABELS)})


In [ ]:
# Load the exact BPE-25k vocabulary and merge list used by the supplied V2 notebook.
BPE_PATH = ASSETS["bpe_en_25k.json"]
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = int(bpe_data["vocab_size"])
assert bpe_vocab_size == 25_000 and len(bpe_itos) == 25_000
bpe_stoi = {token: index for index, token in enumerate(bpe_itos)}
merge_to_id = {}
for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge
    merge_to_id[(bpe_stoi[left_token], bpe_stoi[right_token])] = 261 + merge_index

#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

PAD_TOKEN_ID = tokenizer.pad_token_id
UNK_TOKEN_ID = tokenizer.unk_token_id
BOS_TOKEN_ID = tokenizer.bos_token_id
EOS_TOKEN_ID = tokenizer.eos_token_id


In [ ]:
# These values match the V2 configuration in the supplied Phase 1 notebook.
SEED = 42
N_ENCODER_LAYERS = 4
N_DECODER_LAYERS = 5
N_EMBD = 768
N_HEAD = 12
SOURCE_LENGTH = 512
TARGET_LENGTH = 512
FFN_MULTIPLIER = 4
DROPOUT = 0.0

# Phase 3 training controls. The original V2 pretraining loop is not included.
MAX_LENGTH = 256
BATCH_SIZE = 16
ENCODE_BATCH_SIZE = 32
DOC_BATCH_SIZE = 8
ADAPT_EPOCHS = 3
CONTEXT_EPOCHS = 15
EARLY_STOP_ADAPT = 2
EARLY_STOP_CONTEXT = 4
RANK = 8
ALPHA = 16.0
ADAPTER_LR = 2e-4
HEAD_LR = 8e-4
ENCODER_FULL_LR = 2e-5
CONTEXT_LR = 1e-3
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
KD_TEMPERATURE = 2.0
KD_WEIGHT = 0.25
NUM_WORKERS = 0

assert MAX_LENGTH <= SOURCE_LENGTH
print("V2 config:", {"encoder_layers": N_ENCODER_LAYERS, "decoder_layers": N_DECODER_LAYERS,
                      "width": N_EMBD, "heads": N_HEAD, "source_length": SOURCE_LENGTH})
print("Phase 3 config:", {"max_sentence_tokens": MAX_LENGTH, "rank": RANK, "alpha": ALPHA,
                           "adapt_epochs": ADAPT_EPOCHS, "context_epochs": CONTEXT_EPOCHS})


In [ ]:
# 9. V2 Encoder-Decoder Transformer
# This is our second architecture built from scratch.
# V2 uses:
# - Encoder-decoder Transformer
# - Standard Multi-Head Attention
# - Learned positional embeddings
# - LayerNorm
# - Causal decoder self-attention
# - Decoder cross-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation
#
# We keep the hidden size and attention width close to V1
# so that the comparison mainly reflects the architecture.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# Standard Multi-Head Attention
# ---------------------------------------------------------

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False,
        causal=True
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(x)
        )

        v = self._split_heads(
            self.v_proj(x)
        )

        if past_key_value is not None:

            past_k, past_v = past_key_value

            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # Decoder self-attention is causal.
        # Encoder self-attention is bidirectional.
        #
        # During cached generation, the current token
        # can attend to all previous cached tokens, so
        # no explicit causal mask is needed.
        if causal and past_key_value is None:

            T_q = q.size(-2)
            T_k = k.size(-2)

            causal_mask = torch.tril(
                torch.ones(
                    T_q,
                    T_k,
                    device=x.device,
                    dtype=torch.bool
                )
            )

        else:
            causal_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=causal_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Cross Attention
# ---------------------------------------------------------

class CrossAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        encoder_output
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(encoder_output)
        )

        v = self._split_heads(
            self.v_proj(encoder_output)
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y


# ---------------------------------------------------------
# Standard GELU Feed Forward Network
# ---------------------------------------------------------

class V2FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Encoder Layer
# ---------------------------------------------------------

class EncoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(self, x):

        # Encoder self-attention is bidirectional.
        normalized_x = self.norm1(x)

        attention_output, _ = self.self_attention(
            normalized_x,
            causal=False
        )

        x = x + attention_output

        # Feed-forward network.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ---------------------------------------------------------
# Decoder Layer
# ---------------------------------------------------------

class DecoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.cross_attention = CrossAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        encoder_output,
        past_key_value=None,
        use_cache=False
    ):

        # 1. Decoder causal self-attention.
        attention_output, present = self.self_attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache,
            causal=True
        )

        x = x + attention_output

        # 2. Decoder attends to the encoder output.
        cross_output = self.cross_attention(
            self.norm2(x),
            encoder_output
        )

        x = x + cross_output

        # 3. Feed-forward network.
        x = x + self.ffn(
            self.norm3(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Encoder-Decoder Legal Language Model
# ---------------------------------------------------------

class V2LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_encoder_layers,
        n_decoder_layers,
        n_head,
        source_length,
        target_length,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.source_length = source_length
        self.target_length = target_length

        # Shared token embedding.
        # We also tie this weight to the final LM head.
        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Encoder and decoder use learned positional embeddings.
        self.encoder_position_embedding = nn.Embedding(
            source_length,
            n_embd
        )

        self.decoder_position_embedding = nn.Embedding(
            target_length,
            n_embd
        )

        # Encoder stack.
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_encoder_layers)
        ])

        # Decoder stack.
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_decoder_layers)
        ])

        self.encoder_final_norm = nn.LayerNorm(
            n_embd
        )

        self.decoder_final_norm = nn.LayerNorm(
            n_embd
        )

        # Tied language-model head.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def encode(
        self,
        encoder_input_ids
    ):

        B, T = encoder_input_ids.shape

        positions = torch.arange(
            T,
            device=encoder_input_ids.device
        )

        x = self.token_embedding(
            encoder_input_ids
        )

        x = x + self.encoder_position_embedding(
            positions
        ).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x)

        x = self.encoder_final_norm(x)

        return x

    def decode(
        self,
        decoder_input_ids,
        encoder_output,
        past_key_values=None,
        use_cache=False
    ):

        B, T = decoder_input_ids.shape

        if past_key_values is None:

            position_offset = 0

        else:

            position_offset = (
                past_key_values[0][0].size(-2)
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=decoder_input_ids.device
        )

        x = self.token_embedding(
            decoder_input_ids
        )

        x = x + self.decoder_position_embedding(
            positions
        ).unsqueeze(0)

        presents = [] if use_cache else None

        for layer_index, layer in enumerate(
            self.decoder_layers
        ):

            past = None

            if past_key_values is not None:

                past = past_key_values[
                    layer_index
                ]

            x, present = layer(
                x,
                encoder_output,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:

                presents.append(
                    present
                )

        x = self.decoder_final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

    def forward(
        self,
        encoder_input_ids,
        decoder_input_ids,
        labels=None,
        use_cache=False,
        past_key_values=None,
        encoder_output=None
    ):

        # Encode the source sequence only once.
        if encoder_output is None:

            encoder_output = self.encode(
                encoder_input_ids
            )

        logits, presents = self.decode(
            decoder_input_ids,
            encoder_output,
            past_key_values=past_key_values,
            use_cache=use_cache
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1),
                ignore_index=PAD_TOKEN_ID
            )

        return (
            logits,
            loss,
            presents,
            encoder_output
        )


print("V2 Encoder-Decoder model classes defined.")

In [ ]:
def load_v2_backbone():
    model = V2LegalLM(
        vocab_size=tokenizer.vocab_size,
        n_embd=N_EMBD,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_head=N_HEAD,
        source_length=SOURCE_LENGTH,
        target_length=TARGET_LENGTH,
        ffn_multiplier=FFN_MULTIPLIER,
        dropout=DROPOUT,
    )
    try:
        checkpoint = torch.load(ASSETS["v2_model.pt"], map_location="cpu", weights_only=True)
    except TypeError:
        checkpoint = torch.load(ASSETS["v2_model.pt"], map_location="cpu")
    if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
        raise ValueError("v2_model.pt does not contain the expected model_state_dict wrapper")
    state = checkpoint["model_state_dict"]
    result = model.load_state_dict(state, strict=True)
    config = checkpoint.get("config", {})
    expected = {"architecture": "encoder_decoder", "vocab_size": 25_000, "n_embd": 768,
                "n_encoder_layers": 4, "n_decoder_layers": 5, "n_head": 12,
                "source_length": 512, "target_length": 512, "ffn_multiplier": 4}
    for key, value in expected.items():
        if key in config and config[key] != value:
            raise ValueError(f"Checkpoint config mismatch for {key}: {config[key]} != {value}")
    model.to(DEVICE)
    print("Strict V2 checkpoint load:", result)
    print("Total V2 parameters:", sum(p.numel() for p in model.parameters()))
    del checkpoint, state
    return model

def masked_v2_encoder(backbone, input_ids, attention_mask):
    # V2 encoder forward with valid-key masking; parameter names and weights stay unchanged.
    batch_size, token_count = input_ids.shape
    positions = torch.arange(token_count, device=input_ids.device)
    x = backbone.token_embedding(input_ids)
    x = x + backbone.encoder_position_embedding(positions).unsqueeze(0)
    key_mask = attention_mask.to(torch.bool)[:, None, None, :]
    for layer in backbone.encoder_layers:
        normalized = layer.norm1(x)
        attn = layer.self_attention
        q = attn._split_heads(attn.q_proj(normalized))
        k = attn._split_heads(attn.k_proj(normalized))
        v = attn._split_heads(attn.v_proj(normalized))
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=key_mask,
            dropout_p=(attn.dropout if backbone.training else 0.0),
            is_causal=False,
        )
        x = x + attn.out_proj(attn._merge_heads(y))
        x = x + layer.ffn(layer.norm2(x))
    return backbone.encoder_final_norm(x)

def masked_mean_pool(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

def padded_ids(token_lists, device=DEVICE):
    tensors = [torch.tensor(tokens[:MAX_LENGTH] or [UNK_TOKEN_ID], dtype=torch.long) for tokens in token_lists]
    ids = pad_sequence(tensors, batch_first=True, padding_value=PAD_TOKEN_ID).to(device)
    mask = ids.ne(PAD_TOKEN_ID)
    return ids, mask

# Strict-load and smoke-check the actual checkpoint before any experiment starts.
backbone_probe = load_v2_backbone()
backbone_probe.eval()
probe_ids, probe_mask = padded_ids([[10, 20, 30], [11, 21]])
with torch.inference_mode():
    probe_hidden = masked_v2_encoder(backbone_probe, probe_ids, probe_mask)
    probe_vectors = masked_mean_pool(probe_hidden, probe_mask)
assert probe_hidden.shape == (2, 3, N_EMBD)
assert probe_vectors.shape == (2, N_EMBD)
assert torch.isfinite(probe_vectors).all()

# With no padding, the Phase 3 encoder path must reproduce the original V2 encode path.
plain_ids = torch.tensor([[10, 20, 30]], dtype=torch.long, device=DEVICE)
plain_mask = torch.ones_like(plain_ids, dtype=torch.bool)
with torch.inference_mode():
    original_hidden = backbone_probe.encode(plain_ids)
    masked_hidden = masked_v2_encoder(backbone_probe, plain_ids, plain_mask)
parity_difference = (original_hidden - masked_hidden).abs().max().item()
assert parity_difference < 1e-5, f"Masked path changed the original unpadded V2 encoding: {parity_difference}"

# Padding invariance: a sentence vector must not change when extra masked pad tokens are appended.
single_ids = torch.tensor([[10, 20, 30]], dtype=torch.long, device=DEVICE)
single_mask = torch.ones_like(single_ids, dtype=torch.bool)
padded_probe = torch.tensor([[10, 20, 30, PAD_TOKEN_ID, PAD_TOKEN_ID]], dtype=torch.long, device=DEVICE)
padded_mask = torch.tensor([[1, 1, 1, 0, 0]], dtype=torch.bool, device=DEVICE)
with torch.inference_mode():
    vector_a = masked_mean_pool(masked_v2_encoder(backbone_probe, single_ids, single_mask), single_mask)
    vector_b = masked_mean_pool(masked_v2_encoder(backbone_probe, padded_probe, padded_mask), padded_mask)
pad_difference = (vector_a - vector_b).abs().max().item()
assert pad_difference < 1e-4, f"Padding changed the sentence vector: {pad_difference}"
print("Encoder smoke check passed; original-path max difference:", parity_difference,
      "| masked padding max difference:", pad_difference)
del backbone_probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Tokenize all supplied Phase 3 text with the original V2 byte-level BPE.
for split in ("train", "dev", "test"):
    for domain in ("CL", "IT"):
        for doc in DOCS[split][domain]:
            doc["token_ids"] = []
            for text in doc["text"]:
                tokens = tokenizer.encode(text, add_bos=False, add_eos=False)
                if not tokens:
                    tokens = [UNK_TOKEN_ID]
                doc["token_ids"].append(tokens)

print("BPE token-length and truncation audit at MAX_LENGTH =", MAX_LENGTH)
for split in ("train", "dev", "test"):
    for domain in ("CL", "IT"):
        lengths = [len(tokens) for doc in DOCS[split][domain] for tokens in doc["token_ids"]]
        trunc = sum(length > MAX_LENGTH for length in lengths)
        p95 = float(np.percentile(lengths, 95)) if lengths else 0.0
        p99 = float(np.percentile(lengths, 99)) if lengths else 0.0
        print(f"{domain} {split}: docs={len(DOCS[split][domain])}, sentences={len(lengths)}, "
              f"p95={p95:.1f}, p99={p99:.1f}, max={max(lengths)}, truncated={trunc}")
        if not lengths:
            raise ValueError(f"No sentences in {domain} {split}")

def flatten_labeled(split):
    rows = []
    for domain in ("CL", "IT"):
        for doc in DOCS[split][domain]:
            for sentence_index, (text, tokens, label) in enumerate(zip(doc["text"], doc["token_ids"], doc["labels"])):
                rows.append({
                    "domain": domain, "doc_id": doc["id"], "sentence_index": sentence_index,
                    "text": text, "tokens": tokens[:MAX_LENGTH] or [UNK_TOKEN_ID],
                    "label_id": label_to_id[label],
                })
    return rows

TRAIN_SAMPLES = flatten_labeled("train")
DEV_SAMPLES = flatten_labeled("dev")
assert len(TRAIN_SAMPLES) == 16_845, f"Unexpected training sentence count: {len(TRAIN_SAMPLES)}"
for index, row in enumerate(TRAIN_SAMPLES):
    row["index"] = index
print("Flattened train/dev sentences:", len(TRAIN_SAMPLES), len(DEV_SAMPLES))

# Moderate train-only class weights. The cap prevents three rare examples from receiving an extreme weight.
counts = torch.bincount(torch.tensor([x["label_id"] for x in TRAIN_SAMPLES]), minlength=NUM_LABELS).float()
raw_weights = torch.sqrt(counts.mean() / counts.clamp_min(1.0)).clamp(0.5, 3.0)
CLASS_WEIGHTS = (raw_weights / raw_weights.mean()).to(DEVICE)
print("Train-only class weights:", {id_to_label[i]: round(float(CLASS_WEIGHTS[i]), 3) for i in range(NUM_LABELS)})


In [ ]:
class LowRankAdaptedLinear(nn.Module):
    # LoRA or LoRMA+ wrapper around one frozen V2 Linear layer.
    def __init__(self, base_layer, method, rank=RANK, alpha=ALPHA):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError("Low-rank adapters can only wrap nn.Linear modules")
        self.base_layer = base_layer
        self.method = method
        self.rank = int(rank)
        self.scale = float(alpha) / self.rank
        adapter_device = base_layer.weight.device
        adapter_dtype = base_layer.weight.dtype
        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False
        if method == "lora":
            self.adapter_A = nn.Parameter(torch.empty(self.rank, base_layer.in_features,
                                                     device=adapter_device, dtype=adapter_dtype))
            self.adapter_B = nn.Parameter(torch.zeros(base_layer.out_features, self.rank,
                                                      device=adapter_device, dtype=adapter_dtype))
        elif method == "lorma_plus":
            # In LoRMA+, A acts on W0x in output space and B maps rank back to output size.
            self.adapter_A = nn.Parameter(torch.empty(self.rank, base_layer.out_features,
                                                     device=adapter_device, dtype=adapter_dtype))
            self.adapter_B = nn.Parameter(torch.zeros(base_layer.out_features, self.rank,
                                                      device=adapter_device, dtype=adapter_dtype))
        else:
            raise ValueError(f"Unknown low-rank method: {method}")
        nn.init.kaiming_uniform_(self.adapter_A, a=math.sqrt(5))

    def forward(self, x):
        base_output = self.base_layer(x)
        if self.method == "lora":
            update = F.linear(F.linear(x, self.adapter_A), self.adapter_B)
        else:
            # LoRMA+ implements (I + alpha/r * B A) W0 x; leave the base bias untouched.
            base_without_bias = F.linear(x, self.base_layer.weight, bias=None)
            update = F.linear(F.linear(base_without_bias, self.adapter_A), self.adapter_B)
        return base_output + self.scale * update

def install_low_rank_adapters(backbone, method):
    for layer in backbone.encoder_layers:
        for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
            old = getattr(layer.self_attention, name)
            setattr(layer.self_attention, name, LowRankAdaptedLinear(old, method))
        for name in ("fc1", "fc2"):
            old = getattr(layer.ffn, name)
            setattr(layer.ffn, name, LowRankAdaptedLinear(old, method))

def configure_backbone_trainability(backbone, method):
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    if method == "full_teacher":
        # Full fine-tuning of the Phase 3 encoder path; the autoregressive decoder stays unused/frozen.
        for module in (backbone.token_embedding, backbone.encoder_position_embedding,
                       backbone.encoder_layers, backbone.encoder_final_norm):
            for parameter in module.parameters():
                parameter.requires_grad = True
    elif method == "bitfit":
        for name, parameter in backbone.named_parameters():
            if name.startswith("encoder_") or name.startswith("encoder_layers."):
                if name.endswith(".bias"):
                    parameter.requires_grad = True
    elif method in ("lora", "lorma_plus"):
        install_low_rank_adapters(backbone, method)
    else:
        raise ValueError(f"Unknown adaptation method: {method}")

class SentenceRoleHead(nn.Module):
    def __init__(self, input_size=N_EMBD, hidden_size=384, num_labels=NUM_LABELS, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_labels),
        )
    def forward(self, sentence_vectors):
        return self.net(sentence_vectors)

def trainable_parameter_count(module):
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)

def trainable_state(module):
    return {name: parameter.detach().cpu().clone()
            for name, parameter in module.named_parameters() if parameter.requires_grad}

def restore_trainable_state(module, state):
    parameters = dict(module.named_parameters())
    with torch.no_grad():
        for name, value in state.items():
            if name not in parameters or not parameters[name].requires_grad:
                raise KeyError(f"Saved trainable parameter no longer exists: {name}")
            parameters[name].copy_(value.to(parameters[name].device))

print("PEFT wrappers and classification head defined.")


In [ ]:
class FlatSentenceDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return {"index": row.get("index", index), "domain": row["domain"],
                "tokens": row["tokens"], "label_id": row.get("label_id", -100)}

def collate_sentences(batch):
    token_lists = [row["tokens"][:MAX_LENGTH] or [UNK_TOKEN_ID] for row in batch]
    input_ids, attention_mask = padded_ids(token_lists)
    return {
        "indices": torch.tensor([row["index"] for row in batch], dtype=torch.long),
        "domains": [row["domain"] for row in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": torch.tensor([row["label_id"] for row in batch], dtype=torch.long, device=DEVICE),
    }

def sentence_loader(rows, shuffle=False):
    return DataLoader(FlatSentenceDataset(rows), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, collate_fn=collate_sentences, pin_memory=False)

def macro_f1_fixed(y_true, y_pred):
    if len(y_true) != len(y_pred):
        raise ValueError("Gold and predicted lengths differ")
    scores = []
    for label_id in range(NUM_LABELS):
        tp = sum(1 for gold, pred in zip(y_true, y_pred) if gold == label_id and pred == label_id)
        fp = sum(1 for gold, pred in zip(y_true, y_pred) if gold != label_id and pred == label_id)
        fn = sum(1 for gold, pred in zip(y_true, y_pred) if gold == label_id and pred != label_id)
        denom = 2 * tp + fp + fn
        scores.append((2 * tp / denom) if denom else 0.0)
    return float(sum(scores) / NUM_LABELS)

def domain_macro_metrics(gold_by_domain, pred_by_domain):
    return {domain: macro_f1_fixed(gold_by_domain[domain], pred_by_domain[domain])
            for domain in ("CL", "IT")}

def fixed_mean_score(metrics):
    return (metrics["CL"] + metrics["IT"]) / 2.0

def autocast_context():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.inference_mode()
def evaluate_sentence_model(backbone, head, rows):
    backbone.eval()
    head.eval()
    gold = {"CL": [], "IT": []}
    predicted = {"CL": [], "IT": []}
    for batch in sentence_loader(rows, shuffle=False):
        with autocast_context():
            hidden = masked_v2_encoder(backbone, batch["input_ids"], batch["attention_mask"])
            vectors = masked_mean_pool(hidden, batch["attention_mask"])
            logits = head(vectors)
        pred = logits.argmax(dim=-1).detach().cpu().tolist()
        labels = batch["labels"].detach().cpu().tolist()
        for domain, label_id, prediction in zip(batch["domains"], labels, pred):
            gold[domain].append(label_id)
            predicted[domain].append(prediction)
    metrics = domain_macro_metrics(gold, predicted)
    return metrics, fixed_mean_score(metrics)

@torch.inference_mode()
def collect_sentence_logits(backbone, head, rows):
    backbone.eval()
    head.eval()
    output = [None] * len(rows)
    for batch in sentence_loader(rows, shuffle=False):
        with autocast_context():
            hidden = masked_v2_encoder(backbone, batch["input_ids"], batch["attention_mask"])
            vectors = masked_mean_pool(hidden, batch["attention_mask"])
            logits = head(vectors).float()
        for index, row_logits in zip(batch["indices"].tolist(), logits.cpu()):
            output[index] = row_logits
    if any(x is None for x in output):
        raise RuntimeError("Teacher-logit cache is incomplete")
    return torch.stack(output)

print("Sentence-level data and fixed-13 Macro-F1 functions defined.")


In [ ]:
def train_sentence_stage(method, teacher_logits=None, max_epochs=ADAPT_EPOCHS, seed=SEED):
    set_seed(seed)
    backbone = load_v2_backbone()
    configure_backbone_trainability(backbone, method)
    head = SentenceRoleHead().to(DEVICE)
    backbone_parameters = [p for p in backbone.parameters() if p.requires_grad]
    head_parameters = list(head.parameters())
    backbone_lr = ENCODER_FULL_LR if method == "full_teacher" else ADAPTER_LR
    groups = []
    if backbone_parameters:
        groups.append({"params": backbone_parameters, "lr": backbone_lr})
    groups.append({"params": head_parameters, "lr": HEAD_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    best_score = -1.0
    best_backbone_state = None
    best_head_state = None
    best_metrics = None
    best_epoch = 0
    wait = 0
    history = []
    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        backbone.train()
        head.train()
        running_loss = 0.0
        seen = 0
        for batch in sentence_loader(TRAIN_SAMPLES, shuffle=True):
            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                hidden = masked_v2_encoder(backbone, batch["input_ids"], batch["attention_mask"])
                vectors = masked_mean_pool(hidden, batch["attention_mask"])
                logits = head(vectors)
                hard_loss = F.cross_entropy(logits.float(), batch["labels"], weight=CLASS_WEIGHTS)
                if teacher_logits is not None:
                    t = KD_TEMPERATURE
                    soft_targets = F.softmax(teacher_logits[batch["indices"]].to(DEVICE) / t, dim=-1)
                    soft_loss = F.kl_div(F.log_softmax(logits.float() / t, dim=-1),
                                         soft_targets, reduction="batchmean") * (t * t)
                    loss = (1.0 - KD_WEIGHT) * hard_loss + KD_WEIGHT * soft_loss
                else:
                    loss = hard_loss
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite sentence-stage loss for {method}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            trainable = backbone_parameters + head_parameters
            torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach().float().cpu()) * len(batch["labels"])
            seen += len(batch["labels"])

        dev_metrics, dev_score = evaluate_sentence_model(backbone, head, DEV_SAMPLES)
        epoch_loss = running_loss / max(seen, 1)
        history.append({"epoch": epoch, "train_loss": epoch_loss,
                        "dev_CL_macro_f1": dev_metrics["CL"],
                        "dev_IT_macro_f1": dev_metrics["IT"],
                        "dev_mean_macro_f1": dev_score})
        print(f"{method} sentence epoch {epoch}/{max_epochs} | loss={epoch_loss:.4f} | "
              f"CL={dev_metrics['CL']:.4f} IT={dev_metrics['IT']:.4f} mean={dev_score:.4f}")
        if dev_score > best_score + 1e-8:
            best_score = dev_score
            best_metrics = dict(dev_metrics)
            best_epoch = epoch
            best_backbone_state = trainable_state(backbone)
            best_head_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= (EARLY_STOP_ADAPT if method != "full_teacher" else EARLY_STOP_ADAPT):
                print("Sentence-stage early stop.")
                break

    if best_backbone_state is None:
        raise RuntimeError("No finite development checkpoint was selected")
    restore_trainable_state(backbone, best_backbone_state)
    head.load_state_dict(best_head_state, strict=True)
    elapsed = time.time() - start_time
    trainable_count = trainable_parameter_count(backbone) + sum(p.numel() for p in head.parameters())
    return {"backbone": backbone, "head": head, "metrics": best_metrics,
            "score": best_score, "epoch": best_epoch, "history": history,
            "trainable_params": trainable_count, "seconds": elapsed,
            "best_backbone_state": best_backbone_state}

print("Sentence training stage defined.")


In [ ]:
@torch.inference_mode()
def encode_document_embeddings(backbone, docs, batch_size=ENCODE_BATCH_SIZE):
    backbone.eval()
    result = {}
    for doc in docs:
        token_lists = [tokens[:MAX_LENGTH] or [UNK_TOKEN_ID] for tokens in doc["token_ids"]]
        vectors = []
        for start in range(0, len(token_lists), batch_size):
            ids, mask = padded_ids(token_lists[start:start + batch_size])
            with autocast_context():
                hidden = masked_v2_encoder(backbone, ids, mask)
                pooled = masked_mean_pool(hidden, mask).float()
            vectors.append(pooled.cpu())
        if not vectors:
            raise ValueError(f"Empty test/train document: {doc['id']}")
        matrix = torch.cat(vectors, dim=0)
        if matrix.shape[0] != len(doc["text"]):
            raise RuntimeError(f"Embedding count mismatch for document {doc['id']}")
        result[doc["id"]] = matrix
    return result

def context_examples(split, embeddings):
    examples = []
    for domain in ("CL", "IT"):
        for doc in DOCS[split][domain]:
            labels = None if doc["labels"] is None else [label_to_id[x] for x in doc["labels"]]
            examples.append({"domain": domain, "id": doc["id"],
                             "vectors": embeddings[domain][doc["id"]], "labels": labels})
    return examples

class DocumentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, index):
        return self.examples[index]

def collate_documents(batch):
    lengths = torch.tensor([len(row["vectors"]) for row in batch], dtype=torch.long)
    vectors = pad_sequence([row["vectors"] for row in batch], batch_first=True, padding_value=0.0)
    if batch[0]["labels"] is None:
        labels = None
    else:
        labels = pad_sequence([torch.tensor(row["labels"], dtype=torch.long) for row in batch],
                              batch_first=True, padding_value=-100)
    return {"domains": [row["domain"] for row in batch], "ids": [row["id"] for row in batch],
            "lengths": lengths, "vectors": vectors.to(DEVICE),
            "labels": None if labels is None else labels.to(DEVICE)}

class DocumentContextClassifier(nn.Module):
    def __init__(self, input_size=N_EMBD, hidden_size=256, num_labels=NUM_LABELS, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
    def forward(self, vectors, lengths):
        packed = pack_padded_sequence(vectors, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        sequence, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=vectors.size(1))
        return self.classifier(self.dropout(sequence))

def context_loader(examples, shuffle=False):
    return DataLoader(DocumentDataset(examples), batch_size=DOC_BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, collate_fn=collate_documents,
                      pin_memory=False)

@torch.inference_mode()
def evaluate_context_model(model, examples):
    model.eval()
    gold = {"CL": [], "IT": []}
    predicted = {"CL": [], "IT": []}
    predictions_by_id = {"CL": {}, "IT": {}}
    for batch in context_loader(examples, shuffle=False):
        logits = model(batch["vectors"], batch["lengths"])
        pred = logits.argmax(dim=-1).detach().cpu()
        labels = batch["labels"]
        for i, (domain, doc_id) in enumerate(zip(batch["domains"], batch["ids"])):
            length = int(batch["lengths"][i])
            row_pred = pred[i, :length].tolist()
            row_gold = labels[i, :length].detach().cpu().tolist()
            gold[domain].extend(row_gold)
            predicted[domain].extend(row_pred)
            predictions_by_id[domain][doc_id] = row_pred
    metrics = domain_macro_metrics(gold, predicted)
    return metrics, fixed_mean_score(metrics), predictions_by_id

def train_context_stage(train_examples, dev_examples, seed=SEED + 100):
    set_seed(seed)
    model = DocumentContextClassifier().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONTEXT_LR, weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    best_score = -1.0
    best_state = None
    best_metrics = None
    best_epoch = 0
    wait = 0
    history = []
    for epoch in range(1, CONTEXT_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        steps = 0
        for batch in context_loader(train_examples, shuffle=True):
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch["vectors"], batch["lengths"])
            loss = F.cross_entropy(logits.reshape(-1, NUM_LABELS), batch["labels"].reshape(-1),
                                   weight=CLASS_WEIGHTS, ignore_index=-100)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite document-context loss")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach().float().cpu())
            steps += 1
        dev_metrics, dev_score, _ = evaluate_context_model(model, dev_examples)
        epoch_loss = running_loss / max(steps, 1)
        history.append({"epoch": epoch, "train_loss": epoch_loss,
                        "dev_CL_macro_f1": dev_metrics["CL"],
                        "dev_IT_macro_f1": dev_metrics["IT"],
                        "dev_mean_macro_f1": dev_score})
        print(f"context epoch {epoch}/{CONTEXT_EPOCHS} | loss={epoch_loss:.4f} | "
              f"CL={dev_metrics['CL']:.4f} IT={dev_metrics['IT']:.4f} mean={dev_score:.4f}")
        if dev_score > best_score + 1e-8:
            best_score = dev_score
            best_metrics = dict(dev_metrics)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= EARLY_STOP_CONTEXT:
                print("Context-stage early stop.")
                break
    if best_state is None:
        raise RuntimeError("No context model checkpoint selected")
    model.load_state_dict(best_state, strict=True)
    return {"model": model, "metrics": best_metrics, "score": best_score,
            "epoch": best_epoch, "history": history, "best_state": best_state}

print("Document-context training stage defined.")


## Train the teacher and four PEFT candidates

The teacher is used to produce soft task targets for the LoRMA+KD candidate; it is not counted as an extra submission. The four ZIPs correspond to the PEFT candidates listed below. Each uses the same train/dev split, tokenizer, maximum length, class weights, context classifier, and model-selection rule.

The encoder-stage candidate is selected on development Macro-F1 before its document-context BiLSTM is trained. The final candidate checkpoint is selected again on the document-context development score. This keeps training on gold train labels and keeps all test data strictly inference-only.


In [ ]:
# Train a Phase 3 encoder fine-tuning teacher and cache only its train-set logits for KD.
teacher_result = train_sentence_stage("full_teacher", teacher_logits=None, max_epochs=ADAPT_EPOCHS, seed=SEED)
TEACHER_TRAIN_LOGITS = collect_sentence_logits(teacher_result["backbone"], teacher_result["head"], TRAIN_SAMPLES)
TEACHER_METRICS = {
    "sentence_dev_CL_macro_f1": teacher_result["metrics"]["CL"],
    "sentence_dev_IT_macro_f1": teacher_result["metrics"]["IT"],
    "sentence_dev_mean_macro_f1": teacher_result["score"],
    "selected_epoch": teacher_result["epoch"],
    "trainable_parameters": teacher_result["trainable_params"],
}
print("Teacher selected:", TEACHER_METRICS)
del teacher_result
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
METHODS = [
    ("lora", "lora", False),
    ("lorma_plus", "lorma_plus", False),
    ("bitfit", "bitfit", False),
    ("lorma_plus_kd", "lorma_plus", True),
]

def build_submission_prediction(test_predictions_by_id):
    prediction = {}
    for domain in ("CL", "IT"):
        for doc in DOCS["test"][domain]:
            if doc["id"] not in test_predictions_by_id[domain]:
                raise ValueError(f"Missing prediction for {domain} document {doc['id']}")
            ids = test_predictions_by_id[domain][doc["id"]]
            if len(ids) != len(doc["text"]):
                raise ValueError(f"Sentence count mismatch for {doc['id']}: {len(ids)} != {len(doc['text'])}")
            prediction[doc["id"]] = [id_to_label[int(i)] for i in ids]
    expected_ids = {doc["id"] for domain in ("CL", "IT") for doc in DOCS["test"][domain]}
    if set(prediction) != expected_ids:
        raise ValueError("Prediction document IDs do not exactly match the two test sets")
    allowed = set(label_to_id)
    for doc_id, labels in prediction.items():
        if not all(label in allowed for label in labels):
            raise ValueError(f"Invalid output label in {doc_id}")
    return prediction

def write_submission_zip(prediction, method_name):
    json_path = OUTPUT_DIR / f"prediction_{method_name}.json"
    zip_path = OUTPUT_DIR / f"submission_{method_name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as zf:
        assert zf.namelist() == ["prediction.json"], zf.namelist()
        assert zf.testzip() is None, "ZIP CRC verification failed"
        archived = json.loads(zf.read("prediction.json").decode("utf-8"))
    if archived != prediction:
        raise AssertionError("Archived prediction differs from the generated prediction")
    expected_docs = {doc["id"]: len(doc["text"]) for domain in ("CL", "IT") for doc in DOCS["test"][domain]}
    if {key: len(value) for key, value in archived.items()} != expected_docs:
        raise AssertionError("Submission sentence coverage does not match test data")
    print(f"Verified {zip_path.name}: root prediction.json, {len(archived)} docs, "
          f"{sum(map(len, archived.values()))} sentence labels, {zip_path.stat().st_size:,} bytes")
    return json_path, zip_path

EXPERIMENT_RESULTS = {}
PREDICTION_FILES = {}
ZIP_FILES = {}

for method_index, (run_name, encoder_method, use_kd) in enumerate(METHODS):
    print("\n" + "=" * 80)
    print("RUN:", run_name, "| encoder adaptation:", encoder_method, "| KD:", use_kd)
    sentence_result = train_sentence_stage(
        encoder_method,
        teacher_logits=TEACHER_TRAIN_LOGITS if use_kd else None,
        max_epochs=ADAPT_EPOCHS,
        seed=SEED,
    )
    encoder_seconds = sentence_result["seconds"]
    trainable_count = sentence_result["trainable_params"]

    embedding_cache = {"train": {}, "dev": {}, "test": {}}
    for split in ("train", "dev", "test"):
        for domain in ("CL", "IT"):
            embedding_cache[split][domain] = encode_document_embeddings(
                sentence_result["backbone"], DOCS[split][domain]
            )

    train_docs = context_examples("train", embedding_cache["train"])
    dev_docs = context_examples("dev", embedding_cache["dev"])
    test_docs = context_examples("test", embedding_cache["test"])
    context_result = train_context_stage(train_docs, dev_docs, seed=SEED + 100)
    test_predictions_by_id = {"CL": {}, "IT": {}}
    context_model = context_result["model"]
    context_model.eval()
    with torch.inference_mode():
        for batch in context_loader(test_docs, shuffle=False):
            logits = context_model(batch["vectors"], batch["lengths"])
            pred = logits.argmax(dim=-1).cpu()
            for i, (domain, doc_id) in enumerate(zip(batch["domains"], batch["ids"])):
                length = int(batch["lengths"][i])
                test_predictions_by_id[domain][doc_id] = pred[i, :length].tolist()

    prediction = build_submission_prediction(test_predictions_by_id)
    json_path, zip_path = write_submission_zip(prediction, run_name)
    PREDICTION_FILES[run_name] = json_path
    ZIP_FILES[run_name] = zip_path

    metrics = {
        "method": run_name,
        "encoder_adaptation": encoder_method,
        "uses_knowledge_distillation": use_kd,
        "teacher_sentence_dev": TEACHER_METRICS,
        "sentence_stage_dev": sentence_result["metrics"],
        "sentence_stage_mean_dev_macro_f1": sentence_result["score"],
        "sentence_stage_best_epoch": sentence_result["epoch"],
        "context_stage_dev": context_result["metrics"],
        "context_stage_mean_dev_macro_f1": context_result["score"],
        "context_stage_best_epoch": context_result["epoch"],
        "trainable_encoder_plus_sentence_head_parameters": trainable_count,
        "v2_total_parameters": 95_556_864,
        "trainable_percent_of_v2": 100.0 * trainable_count / 95_556_864,
        "encoder_stage_seconds": encoder_seconds,
        "test_documents": len(prediction),
        "test_sentences": sum(len(x) for x in prediction.values()),
        "prediction_json": str(json_path),
        "submission_zip": str(zip_path),
        "sentence_training_history": sentence_result["history"],
        "context_training_history": context_result["history"],
    }
    EXPERIMENT_RESULTS[run_name] = metrics

    # Keep a compact adapter/context bundle; the large Phase 1 V2 checkpoint is not duplicated.
    bundle = {
        "method": run_name,
        "base_checkpoint": "v2_model.pt",
        "label_to_id": label_to_id,
        "max_length": MAX_LENGTH,
        "rank": RANK,
        "alpha": ALPHA,
        "encoder_trainable_state": sentence_result["best_backbone_state"],
        "sentence_head_state_dict": {k: v.detach().cpu() for k, v in sentence_result["head"].state_dict().items()},
        "context_state_dict": context_result["best_state"],
        "dev_metrics": {"CL": context_result["metrics"]["CL"], "IT": context_result["metrics"]["IT"],
                        "mean": context_result["score"]},
    }
    torch.save(bundle, OUTPUT_DIR / f"adapter_bundle_{run_name}.pt")

    print("Selected context development metrics:", context_result["metrics"],
          "mean=", round(context_result["score"], 4))
    del sentence_result, context_result, context_model, embedding_cache, train_docs, dev_docs, test_docs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nCompleted candidate runs:", list(EXPERIMENT_RESULTS))


In [ ]:
# Select exactly by mean CL/IT development Macro-F1; test labels are not involved.
comparison = {
    "selection_metric": "unweighted mean of fixed-13-label CL and IT development Macro-F1",
    "teacher": TEACHER_METRICS,
    "methods": EXPERIMENT_RESULTS,
}
best_method = max(EXPERIMENT_RESULTS, key=lambda name: EXPERIMENT_RESULTS[name]["context_stage_mean_dev_macro_f1"])
best_source_zip = ZIP_FILES[best_method]
selected_zip = OUTPUT_DIR / "submission.zip"
shutil.copyfile(best_source_zip, selected_zip)
comparison["selected_method_by_development_score"] = best_method
comparison["selected_submission_zip"] = str(selected_zip)

with open(OUTPUT_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(selected_zip, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    assert zf.testzip() is None
print("\nSelected method:", best_method)
print("Selected submission:", selected_zip)
print("Per-method ZIPs:", {name: str(path) for name, path in ZIP_FILES.items()})
print("Comparison record:", OUTPUT_DIR / "model_comparison.json")


## Expected deliverables after a successful Kaggle run

- `submission.zip` — one selected candidate, selected by development score; contains only root-level `prediction.json`.
- `submission_lora.zip`, `submission_lorma_plus.zip`, `submission_bitfit.zip`, `submission_lorma_plus_kd.zip` — four PEFT candidate submissions with the same required output schema.
- `prediction_<method>.json` — unpacked predictions for audit.
- `model_comparison.json` — teacher and candidate development metrics, epochs, trainable parameter counts, durations, and artifact locations.
- `adapter_bundle_<method>.pt` — compact trainable adapter/context state; requires the original `v2_model.pt` to reload.

Codabench's public score is not knowable from the development results. The 0.4311 score from the existing V1 context submission is a separate backbone/run and must not be presented as a V2 result. Wait for this notebook to finish and submit its `submission.zip` before comparing hidden scores.


## References

1. Hu, E. J. et al. (2021). [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685). *arXiv:2106.09685*.
2. Bihany, H., Patel, S., and Modi, A. (2025). [LoRMA: Low-Rank Multiplicative Adaptation for LLMs](https://aclanthology.org/2025.findings-acl.527/). *Findings of ACL 2025*, 10113–10133.
3. Ben Zaken, E., Goldberg, Y., and Ravfogel, S. (2022). [BitFit: Simple Parameter-efficient Fine-tuning for Transformer-based Masked Language-models](https://aclanthology.org/2022.acl-short.1/). *ACL 2022*, 1–9.
4. Hinton, G., Vinyals, O., and Dean, J. (2015). [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531). *NIPS Deep Learning Workshop*.
5. Malik, V. et al. (2022). [Semantic Segmentation of Legal Documents via Rhetorical Roles](https://aclanthology.org/2022.nllp-1.13/). *NLLP 2022*, 153–171.
6. Jain, D., Borah, M. D., and Biswas, A. (2023). [NITS_Legal at SemEval-2023 Task 6: Rhetorical Roles Prediction of Indian Legal Documents via Sentence Sequence Labeling Approach](https://aclanthology.org/2023.semeval-1.103/). *SemEval 2023*, 751–757.
7. scikit-learn developers. [F1 score documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html). Macro averaging is the unweighted mean across labels; this notebook explicitly includes all 13 labels.

**Project-file sources:** `v2.ipynb` cells 7–8 and 26 define the V2 model/configuration; cells 18, 41, 49, and 61–65 document pretraining data handling, training/evaluation, checkpoint saving, Phase 1 text generation, and the unrelated Phase 1 submission format. Local files `training_history.json`, `v2_model.pt`, and `prediction.json` were inspected directly. The Phase 3 data and submission format are validated against the six supplied CL/IT JSON files and `label_vocab.json`.
